In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def generate_synthetic_data(num_rows=50000, output_file='synthetic_data.csv'):
    """
    Generates a synthetic dataset for the Credit Card Routing project 
    mimicking the structure of the original proprietary data.
    """
    
    # 1. Define Categorical Options based on Case Description
    psps = ['Moneycard', 'Goldcard', 'UK_Card', 'Simplecard']
    cards = ['Master', 'Visa', 'Diners']
    countries = ['Germany', 'Switzerland', 'Austria'] # Common DACH region countries for this specific case study
    
    # 2. Generate Random Data
    print(f"Generating {num_rows} rows of synthetic data...")
    
    # Timestamp: Random times within the last year
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2024, 1, 1)
    time_range = end_date - start_date
    # Generate random seconds offsets
    random_seconds = np.random.randint(0, int(time_range.total_seconds()), num_rows)
    timestamps = [start_date + timedelta(seconds=int(s)) for s in random_seconds]
    
    # Create DataFrame
    df = pd.DataFrame({
        'tmsp': timestamps,
        'country': np.random.choice(countries, num_rows),
        'amount': np.random.randint(5, 500, num_rows), # Random amounts between 5 and 500
        'success': np.random.choice([0, 1], num_rows, p=[0.7, 0.3]), # 70% failure rate to mimic "high failure rate" problem
        'PSP': np.random.choice(psps, num_rows),
        '3D_secured': np.random.choice([0, 1], num_rows, p=[0.8, 0.2]), # 20% 3D secured
        'card': np.random.choice(cards, num_rows)
    })
    
    # Sort by timestamp to look realistic
    df = df.sort_values('tmsp').reset_index(drop=True)
    
    # ---------------------------------------------------------
    # OPTIONAL: LOGIC TO CALCULATE FEES (Based on Section 1.2)
    # ---------------------------------------------------------
    # If your notebook expects a 'total_fees' column immediately, 
    # this block calculates it based on the rules provided.
    
    def calculate_fee(row):
        # Fee structure: (Success Fee, Failed Fee)
        fees = {
            'Moneycard': (5, 2),
            'Goldcard': (10, 5),
            'UK_Card': (3, 1),
            'Simplecard': (1, 0.5)
        }
        
        success_fee, fail_fee = fees.get(row['PSP'], (0,0))
        return success_fee if row['success'] == 1 else fail_fee

    # Apply the fee calculation
    # Note: Rename 'total_fees' to whatever your notebook expects if different
    df['total_fees'] = df.apply(calculate_fee, axis=1) 
    
    # ---------------------------------------------------------

    # 3. Save to CSV
    df.to_csv(output_file, index=False)
    print(f"Data saved to {output_file}. Columns: {list(df.columns)}")
    return df

if __name__ == "__main__":
    # Run generation
    generate_synthetic_data()